In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [2]:
CSV_PATH = '/Users/shubhbhateja/College/minor_project/NeoJaundice/chd_jaundice_published_2.csv'
IMAGE_DIR = '/Users/shubhbhateja/College/minor_project/images'

In [3]:
def prepare_image_multimodal(img_path):
    img = cv2.imread(img_path)
    if img is None: return None
    
    # White Balance & Adaptive Crop
    balanced = white_balance(img) # Uses your existing Gray World function
    rgb_img = cv2.cvtColor(balanced, cv2.COLOR_BGR2RGB)
    skin_patch = adaptive_crop_skin(rgb_img) # Uses your YCrCb function
    
    resized_rgb = cv2.resize(skin_patch, (128, 128))
    resized_hsv = cv2.cvtColor(resized_rgb, cv2.COLOR_RGB2HSV).astype('float32')
    
    # Correct Normalization: Hue is 0-179 in OpenCV
    hsv_norm = np.zeros_like(resized_hsv)
    hsv_norm[:,:,0] = resized_hsv[:,:,0] / 179.0
    hsv_norm[:,:,1] = resized_hsv[:,:,1] / 255.0
    hsv_norm[:,:,2] = resized_hsv[:,:,2] / 255.0
    
    rgb_norm = resized_rgb.astype('float32') / 255.0
    
    # Combine to 6-channel sequence (16384, 6)
    combined = np.concatenate([rgb_norm.reshape(-1, 3), hsv_norm.reshape(-1, 3)], axis=-1)
    return combined

In [4]:
def build_multimodal_1dcnn():
    # Visual Branch
    img_input = layers.Input(shape=(16384, 6), name="skin_sequence_input")
    x = layers.Conv1D(32, 3, padding='same')(img_input)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(4)(x)
    
    x = layers.Conv1D(64, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(4)(x)
    
    x = layers.Conv1D(128, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    # Global Pooling prevents the "Black Box" overfitting
    x = layers.GlobalAveragePooling1D()(x)
    vis_feat = layers.Dense(64, activation='relu')(x)
    vis_feat = layers.Dropout(0.4)(vis_feat)

    # Clinical Branch
    clin_input = layers.Input(shape=(3,), name="clinical_input")
    c = layers.Dense(16, activation='relu')(clin_input)
    clin_feat = layers.Dense(8, activation='relu')(c)

    # Fusion
    merged = layers.Concatenate()([vis_feat, clin_feat])
    head = layers.Dense(32, activation='relu')(merged)
    output = layers.Dense(1, activation='linear', name="tsb_prediction")(head)

    model = models.Model(inputs=[img_input, clin_input], outputs=output)
    
    # Huber loss is more stable for jaundice regression
    model.compile(optimizer=tf.keras.optimizers.Adam(0.0005), loss='huber', metrics=['mae'])
    return model

model = build_multimodal_1dcnn()

In [5]:
# --- 5. TRAINING ---
# Weight high-TSB samples (>12mg/dL) more heavily
weights = np.where(y_train_scaled > 0.5, 2.0, 1.0)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6),
    tf.keras.callbacks.TerminateOnNaN()
]

history = model.fit(
    x={'skin_sequence_input': X_i_train, 'clinical_input': X_c_train},
    y=y_train_scaled,
    sample_weight=weights,
    validation_data=({'skin_sequence_input': X_i_val, 'clinical_input': X_c_val}, y_val_scaled),
    epochs=100,
    batch_size=32,
    callbacks=callbacks
)


NameError: name 'y_train_scaled' is not defined

In [ ]:

# --- 6. INVERSE EVALUATION (mg/dL) ---
preds_scaled = model.predict({'skin_sequence_input': X_i_test, 'clinical_input': X_c_test})
preds_real = scaler_y.inverse_transform(preds_scaled)
y_real = scaler_y.inverse_transform(y_test_scaled.reshape(-1, 1))

print(f"Final Test MAE: {np.mean(np.abs(preds_real - y_real)):.2f} mg/dL")